In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms


In [3]:
# define the network
from vit_pytorch import ViT
model = ViT(
    image_size = 256,
    patch_size = 32,
    num_classes = 10,
    dim = 512,
    depth = 4,
    heads = 4,
    mlp_dim = 512,
    dropout = 0.1,
    emb_dropout = 0.1
)

In [9]:
# define datasets cifar10
transform = transforms.Compose([
    transforms.Resize(256),
    
    transforms.ToTensor()
])
trainset = datasets.CIFAR10(root='./data', train=True, download=False, transform=transform)

In [10]:
trainset[0][0].shape

torch.Size([3, 256, 256])

In [11]:
# define dataloaders
trainloader = DataLoader(trainset, batch_size=8, shuffle=True)
# define device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
# define optimizer and loss
optimizer = optim.AdamW(model.parameters(), lr=1e-3)
cross_entropy_loss = nn.CrossEntropyLoss()


In [13]:
# define training params
epochs = 5
saving_interval_steps = 5000
logs_interval_steps = 100
saved_dir = "checkpoints"

In [ ]:
# write the training loop
step = 0
model.train()
for epoch in range(epochs):
    for _, data in enumerate(trainloader, 1):
        inputs, labels = data
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = cross_entropy_loss(outputs, labels)
        loss.backward()
        optimizer.step()
        if step % logs_interval_steps == 0 and step!=0 :
            print(f'epoch: {epoch}, batch: {step}, loss: {loss.item()}')
        if step % saving_interval_steps == 0 and step!=0 :
            torch.save(model.state_dict(), f'{saved_dir}/vit_{epoch}_{step}.pth')
        step += 1
        
        

In [4]:
# speed testing
import time
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
model.load_state_dict(torch.load('checkpoints/vit_4_30000.pth', map_location='cpu'))
model.eval()
inputs = torch.randn(1, 3, 256, 256).to(device)
print(device)
outputs = model(inputs)
total_time = 0.0
for _ in range(10000):
    start_time = time.time()
    outputs = model(inputs)
    total_time += time.time() - start_time
print(f"10000 inference took: {total_time} seconds")


cuda
